# 07 — AI Insight + Verifier (Phase 8)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 8: import และเรียกใช้ฟังก์ชันจาก `src/rri/insight/`
เท่านั้น — ไม่มี business logic ในเซลล์ ตาม CLAUDE.md N2

**แนวคิดหลักของ Phase 8:** LLM เก่งเรื่องแต่งประโยคให้อ่านง่าย แต่ไม่น่าเชื่อถือเรื่องจำตัวเลขแม่นยำ
ดังนั้นเราให้ AI ทำแค่หน้าที่แรก (เขียนบรรยาย) ส่วนหน้าที่สอง (ตรวจตัวเลข) ให้ verifier ที่เป็น
โค้ดล้วน ไม่ใช้ AI ตรวจแทน แล้วรายงาน hallucination rate เป็นตัวเลขที่วัดได้จริง

ลำดับขั้นตอนที่โน้ตบุ๊กนี้แสดง:

1. **Facts contract** (`rri.insight.facts`) — ดึงชุดตัวเลขที่ verified ได้จาก warehouse
2. **Constrained generation** (`rri.insight.generate`) — ให้ Claude แต่งบรรยายจากตัวเลขชุดนั้นเท่านั้น
3. **Verifier** (`rri.insight.verify`) — ตรวจทุกตัวเลขในบรรยายย้อนกลับไปที่ warehouse
4. **การสาธิต planted error** — จงใจปลอมตัวเลขในบรรยาย เพื่อพิสูจน์ว่า verifier จับได้จริง
   (**ตัวเลข hallucination rate ในส่วนนี้มาจากข้อความที่ถูกดัดแปลงโดยเจตนา ไม่ใช่ผลจากการทำงานปกติ**)
5. **สรุปผลจริงจาก cache** — hallucination rate โดยรวมจากทุก scope ที่ generate จริง
   (`data/60_insights/insights_cache.json`, Phase 8 Step 4)


## 0. Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))


from rri.insight.cache import CACHE_PATH, load_cache
from rri.insight.facts import InsightScope, build_facts, get_connection
from rri.insight.generate import GeneratedInsight, generate_insight
from rri.insight.verify import verify_insight
from rri.utils.logging import configure_logging

configure_logging()
con = get_connection()

## 1. Facts contract — ตัวเลขที่ verified ได้ (Step 1)

`build_facts` ดึงชุดตัวเลขปิด (closed set) จาก `fact_accident` / `fact_underwriting_monthly`
สำหรับ scope หนึ่ง ๆ (จังหวัด/ช่วงเวลา) — ทุกตัวเลขมี key, value, unit, description ที่ตายตัว
นี่คือ **แหล่งความจริงเดียว** ที่ AI จะได้รับอนุญาตให้พูดถึง ไม่มีตัวเลขไหนถูกคำนวณโดย AI เอง


In [2]:
scope = InsightScope(
    label="National overview, 2025",
    start_date=__import__("datetime").date(2025, 1, 1),
    end_date=__import__("datetime").date(2025, 12, 31),
)
facts = build_facts(con, scope)
print(f"Scope: {facts.scope_label}  ({len(facts.facts)} facts)")
for f in facts.facts:
    print(f"  {f.key:45s} = {f.value:>12}  [{f.unit}]")

2026-08-25T09:56:03.522206Z [info     ] insight.facts.built            end_date=2025-12-31 n_facts=14 run_id=ee99a592f30f4b12b49fec390fa66ba5 scope='National overview, 2025' start_date=2025-01-01


Scope: National overview, 2025  (14 facts)
  accident_count                                =      23715.0  [accidents]
  fatality_count                                =       2783.0  [fatalities]
  total_injury_count                            =      19475.0  [injuries]
  severity_index_coarse                         =       6.4597  [severity index (weighted fatalities+injuries per accident)]
  accident_count_yoy_pct_change                 =        -0.99  [percent]
  severity_index_coarse_yoy_pct_change          =        -5.23  [percent]
  loss_ratio                                    =        66.43  [percent]
  claim_frequency                               =         0.06  [claims per policy-year]
  top_risk_province_1_severity_index            =       13.966  [severity index (weighted fatalities+injuries per accident)]
  top_risk_province_1_accident_count            =        144.0  [accidents]
  top_risk_province_2_severity_index            =      12.3569  [severity index (weighted fa

## 2. Constrained generation — ให้ AI แต่งบรรยาย (Step 2)

`generate_insight` เรียก Claude ผ่าน tool call ที่บังคับ (`submit_insight`) เพื่อให้ได้ narrative
กับรายการ citation ที่มีโครงสร้างแน่นอน system prompt สั่งชัดเจนว่าใช้ได้เฉพาะตัวเลขใน facts
payload เท่านั้น ห้ามคำนวณ ห้ามปัดเศษต่าง ห้ามเดาตัวเลขใหม่


In [3]:
insight: GeneratedInsight = generate_insight(facts)
print(insight.narrative)

HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


2026-08-25T09:56:09.477665Z [info     ] insight.generate.completed     model=claude-sonnet-5 n_self_reported_citations=14 run_id=ee99a592f30f4b12b49fec390fa66ba5 scope='National overview, 2025'


Nationally in 2025, 23,715 accidents resulted in 2,783 fatalities and 19,475 injuries, yielding a severity index of 6.46 — down 5.23% year-over-year, alongside a 0.99% decline in accident counts. The simulated insurance portfolio posted a 66.43% loss ratio with a claim frequency of 0.06 claims per policy-year. Mukdahan (severity index 13.97, 144 accidents), Phitsanulok (12.36, 197 accidents), and Prachin Buri (11.71, 249 accidents) stand out as the top three highest-severity provinces.


## 3. Verifier — ตรวจย้อนกลับจากข้อความจริง (Step 3, ผลการทำงานปกติ)

`verify_insight` **ไม่เชื่อ** citation ที่ AI รายงานเอง — มัน parse ตัวเลขออกจากตัว narrative
(ข้อความที่คนอ่านจริง ๆ) แล้วเทียบกับค่าจริงใน facts payload ภายใน tolerance 0.5%

ผลลัพธ์ด้านล่างนี้คือ **การทำงานปกติของระบบ** (normal operation) — ไม่ใช่กรณีสาธิต error


In [4]:
record = verify_insight(insight, facts)
print(f"Total numbers checked : {record.total_numbers}")
print(f"Matched               : {record.matched}")
print(f"Flagged               : {record.flagged}")
print(f"Hallucination rate    : {record.hallucination_rate:.1%}  <-- ผลจริง จากการทำงานปกติ")

2026-08-25T09:56:09.483287Z [info     ] insight.verify.completed       flagged=0 hallucination_rate=0.0 matched=14 run_id=ee99a592f30f4b12b49fec390fa66ba5 scope='National overview, 2025' total_numbers=14


Total numbers checked : 14
Matched               : 14
Flagged               : 0
Hallucination rate    : 0.0%  <-- ผลจริง จากการทำงานปกติ


## 4. การสาธิต Planted Error — พิสูจน์ว่า verifier จับได้จริง

ส่วนนี้จงใจปลอมตัวเลขในบรรยายจริงด้านบน (เปลี่ยน `loss_ratio` ให้ผิดจากค่าจริง) เพื่อพิสูจน์ว่า
verifier ตรวจจับความผิดปกติได้จริง ไม่ใช่แค่ rubber-stamp สิ่งที่ AI ยืนยันเอง

> ⚠️ **hallucination rate ที่ได้จากเซลล์นี้มาจากข้อความที่ถูกดัดแปลงโดยเจตนา (deliberately
> corrupted) เพื่อสาธิตกลไกเท่านั้น ไม่ใช่ตัวเลขที่สะท้อนการทำงานจริงของระบบ** — ตัวเลขจริงอยู่ใน
> ส่วนที่ 3 (0% ในตัวอย่างนี้) และส่วนที่ 5 (สรุปรวมทุก scope ที่ generate จริง)


In [5]:
from rri.insight.generate import CitedFact

loss_ratio_fact = facts.as_dict()["loss_ratio"]
true_value = loss_ratio_fact.value
planted_value = round(true_value + 10.0, 1)  # deliberately wrong, +10 pp off

true_str = f"{true_value:g}%"
planted_str = f"{planted_value:g}%"
tampered_narrative = insight.narrative.replace(true_str, planted_str, 1)

print(f"Real loss_ratio fact value : {true_value}%")
print(f"Planted (wrong) value      : {planted_value}%")
print()
print("Tampered narrative:")
print(tampered_narrative)

tampered_insight = GeneratedInsight(
    narrative=tampered_narrative,
    self_reported_citations=[
        CitedFact(fact_key="loss_ratio", value_as_used=planted_str),
        *insight.self_reported_citations,
    ],
    model=insight.model,
    scope_label=insight.scope_label,
)

tampered_record = verify_insight(tampered_insight, facts)
print()
print(f"Total numbers checked : {tampered_record.total_numbers}")
print(f"Flagged               : {tampered_record.flagged}")
print(
    f"Hallucination rate    : {tampered_record.hallucination_rate:.1%}"
    "  <-- from DELIBERATELY CORRUPTED text, not normal operation"
)
for m in tampered_record.matches:
    if m.status != "matched":
        print(f"  FLAGGED: '{m.raw_text}' (fact={m.matched_fact_key}, true value={m.fact_value})")

Real loss_ratio fact value : 66.43%
Planted (wrong) value      : 76.4%

Tampered narrative:
Nationally in 2025, 23,715 accidents resulted in 2,783 fatalities and 19,475 injuries, yielding a severity index of 6.46 — down 5.23% year-over-year, alongside a 0.99% decline in accident counts. The simulated insurance portfolio posted a 76.4% loss ratio with a claim frequency of 0.06 claims per policy-year. Mukdahan (severity index 13.97, 144 accidents), Phitsanulok (12.36, 197 accidents), and Prachin Buri (11.71, 249 accidents) stand out as the top three highest-severity provinces.
2026-08-25T09:56:09.491167Z [info     ] insight.verify.completed       flagged=1 hallucination_rate=0.0714 matched=13 run_id=ee99a592f30f4b12b49fec390fa66ba5 scope='National overview, 2025' total_numbers=14



Total numbers checked : 14
Flagged               : 1
Hallucination rate    : 7.1%  <-- from DELIBERATELY CORRUPTED text, not normal operation
  FLAGGED: '76.4%' (fact=loss_ratio, true value=66.43)


## 5. สรุปผลจริงจากทุก scope ที่ generate จริง (Step 4 cache)

`data/60_insights/insights_cache.json` เก็บผลจากการรัน `scripts/generate_insight_cache.py`
ครั้งเดียวจริงกับ warehouse — ครอบคลุม 5 scope คงที่ (national overview 2 ช่วงเวลา + 3 จังหวัดที่
เสี่ยงสูงสุดจาก scope national_2025 เอง) นี่คือ **hallucination rate โดยรวมที่ใช้รายงานในรายงาน
ฉบับสมบูรณ์** ไม่ใช่ตัวเลขจากการสาธิต error ด้านบน


In [6]:
cache = load_cache()
print(f"Cache path      : {CACHE_PATH}")
print(f"Generated at    : {cache.generated_at}")
print(f"Scopes cached   : {len(cache.entries)}")
print()

total_numbers = sum(e.verification.total_numbers for e in cache.entries)
total_flagged = sum(e.verification.flagged for e in cache.entries)
aggregate_rate = (total_flagged / total_numbers) if total_numbers else 0.0

for e in cache.entries:
    v = e.verification
    print(
        f"  {e.scope_key:28s} numbers={v.total_numbers:2d}  flagged={v.flagged}  "
        f"rate={v.hallucination_rate:.1%}"
    )

print()
print(f"AGGREGATE across all {len(cache.entries)} real, non-corrupted scopes:")
print(f"  {total_flagged}/{total_numbers} numbers flagged")
print(f"  Hallucination rate = {aggregate_rate:.1%}  (report-ready figure)")

Cache path      : /Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/60_insights/insights_cache.json
Generated at    : 2026-08-25T09:46:13.018966+00:00
Scopes cached   : 5

  national_2025                numbers=14  flagged=0  rate=0.0%
  national_2019_2025           numbers=14  flagged=0  rate=0.0%
  province_mukdahan_2025       numbers= 8  flagged=0  rate=0.0%
  province_phitsanulok_2025    numbers= 8  flagged=0  rate=0.0%
  province_prachin_buri_2025   numbers= 8  flagged=0  rate=0.0%

AGGREGATE across all 5 real, non-corrupted scopes:
  0/52 numbers flagged
  Hallucination rate = 0.0%  (report-ready figure)


## 6. สรุป

- **Facts contract** (`rri.insight.facts`) เป็นแหล่งความจริงเดียว — AI ไม่คำนวณตัวเลขเอง
- **Constrained generation** (`rri.insight.generate`) บังคับให้ AI อ้างอิงเฉพาะตัวเลขที่ได้รับ
- **Verifier** (`rri.insight.verify`) เป็นโค้ดล้วน ไม่ใช้ AI ตรวจ AI — parse ตัวเลขจาก narrative
  ข้อความจริงที่คนอ่าน แล้วเทียบกับ warehouse ภายใน tolerance 0.5%
- การสาธิต planted error (ส่วนที่ 4) พิสูจน์ว่ากลไกนี้จับความผิดพลาดได้จริง ไม่ใช่แค่ทฤษฎี —
  แต่ตัวเลข hallucination rate ตรงนั้นมาจากข้อความที่ถูกดัดแปลงโดยเจตนา **ต้องไม่นำไปปนกับ**
  ตัวเลขผลจริงในส่วนที่ 3 และ 5
- ผลจริงจากการ generate ทั้ง 5 scope (Step 4 cache): **0 ใน 52 ตัวเลขถูก flag → hallucination
  rate 0.0%** — ตัวเลขนี้คือค่าที่ใช้รายงานในรายงานฉบับสมบูรณ์

รายละเอียดเพิ่มเติม: `docs/ai_usage_log.md`, `CLAUDE.md` §12 (Phase 8)
